# 数据统计

In [1]:
import pandas as pd
import json
import networkx as nx
import matplotlib.pyplot as plt
import math

from collections import defaultdict

In [2]:
import sys
sys.path.append('..')

## 数据处理

In [3]:
def id_process(df, is_ost=False):
    # 曲目，专辑添加唯一id
    df = df.copy()
    df['song_id_unique'] = 'song' + df['song_id'].astype(str)
    # df_album_fixed = df[['album_id', 'album_fixed']].drop_duplicates(subset=['album_fixed'], keep='first').reset_index(drop=True)
    # df_album_fixed['album_id_unique'] = 'album' + df_album_fixed['album_id'].astype(int).astype(str)
    # df_album_fixed = df_album_fixed[['album_id_unique', 'album_fixed']]
    df['song_year'] = df['publish_date'].str.split('-').str[0]
    if is_ost:
        df['album_fixed'] = df['song_year'] + '年'
        df['album_id_unique'] = 'album' + df['song_year']
        df['legend_type'] = df['song_year'] + '年'
    else:
        df['album_fixed'] = df['album_name']
        df['album_id_unique'] = 'album' + df['album_id'].astype(str)
        df['legend_type'] = df['album_name']
    # df['legend_order'] = df['legend_type']
    # 词唯一id
    df_words = df[['word', 'pos']].drop_duplicates().reset_index(drop=False)
    df_words['word_id'] = 'word' + df_words['index'].astype(str)
    df_words = df_words.drop('index', axis=1).reset_index(drop=True)
    df = df.merge(df_words, on=['word', 'pos'], how='left')
    return df

# 分词数据
名词： n, w
动词： v
形容词： v

In [ ]:
def word_count_by_pos(df, pos, words_num=30, is_starts_with=True):
    if is_starts_with:
        df_sub = df[df['pos'].str.startswith(
            pos, na=False)]
        # 歌曲数
        word_cnt = df_sub.groupby('word')['song_id'].nunique().reset_index().rename(columns={'song_id': 'song_num'})
        word_sum = df[df['pos'].str.startswith(
            pos, na=False)].groupby('word')['freq'].sum().reset_index()
    else:
        word_cnt = df[df['pos']==pos]['word'].groupby('word')['song_id'].nunique().reset_index().rename(columns={'song_id': 'song_num'})
        word_sum = df[df['pos']==pos].groupby('word')['freq'].sum().reset_index()
    if words_num:
        word_cnt = word_cnt.sort_values(by='song_num', ascending=False)
        res = word_cnt.head(words_num).merge(word_sum, on='word', how='left')
    else:
        res = word_cnt.merge(word_sum, on='word', how='left')
    res['order'] = 100 - res.index
    # res = res.rename(columns={
    #     'count': 'songs_num',
    # })
    return res

# 词图

## 二分图布局

In [5]:
# 120度弧线布局
def generate_symmetric_bipartite_layout(nodes):
    coords = {}
    
    # --- 1. 参数定义 ---
    # 定义两条对称弧线的几何参数
    arc_radius = 800
    arc_span = math.pi / 1.5  # 约 120 度
    # 计算弧线的垂直跨度 (用于决定 Word 长度)
    arc_vertical_span = 2 * arc_radius * math.sin(arc_span / 2)
    
    # --- 2. Word 节点 (画布中央直线) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'], key=lambda x: x['degree'], reverse=True)
    num_words = len(word_nodes)
    
    # 长度为弧线跨度的 90%
    target_word_height = arc_vertical_span * 0.9
    word_y_gap = target_word_height / (num_words - 1) if num_words > 1 else 0

    for i, node in enumerate(word_nodes):
        # 中心向两端扩散逻辑: 0->0, 1->1, 2->-1, 3->2...
        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1
        if i == 0: direction = 0
        
        # Word 位于 x=0，且在 y 轴居中
        coords[node['id']] = (0, round(rank * direction * word_y_gap, 2))

    # --- 3. Song 节点 (两侧对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    # 将 Song 平均分为两组：左侧弧和右侧弧
    song_columns = [song_nodes[:mid_idx], song_nodes[mid_idx:]]
    
    # 弧线布局配置：[左侧弧, 右侧弧]
    # 左侧弧圆心在正 X，向左弯曲；右侧弧圆心在负 X，向右弯曲
    configs = [
        {"center_x": 0, "direction": -1}, # 右侧弧 (位于 Word 右侧)
        {"center_x": 0, "direction": 1}  # 左侧弧 (位于 Word 左侧)
    ]

    for col_idx, col_items in enumerate(song_columns):
        config = configs[col_idx]
        num_in_col = len(col_items)
        if num_in_col == 0: continue
        
        for row_idx, node in enumerate(col_items):
            # 从上到下均匀分布角度
            if num_in_col > 1:
                angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span
            else:
                angle = 0
            
            # 计算坐标
            # cos(angle) 决定 X 偏移，direction 决定是在圆心左侧还是右侧
            x = config["center_x"] + config["direction"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))
            
    return coords


In [6]:
# 112度弧形布局
def generate_embracing_layout(nodes):
    coords = {}
    
    # --- 1. 几何参数设定 ---
    arc_radius = 800           # 半径
    arc_span = math.pi / 1.6   # 弧度张角 (约112度)
    # 弧开口端点距离中心直线的水平间距
    horizontal_gap = 300       
    
    # 计算弧线端点的 Y 轴跨度 (用于对齐 Word)
    # y = r * sin(theta)
    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height
    
    # --- 2. Word 节点 (居中直线) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'], key=lambda x: x['degree'], reverse=True)
    num_words = len(word_nodes)
    
    # 长度为弧垂直跨度的 90%
    target_word_height = arc_total_height * 0.9
    word_y_gap = target_word_height / (num_words - 1) if num_words > 1 else 0

    for i, node in enumerate(word_nodes):
        # 中心扩散排序
        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1
        if i == 0: direction = 0
        coords[node['id']] = (0, round(rank * direction * word_y_gap, 2))

    # --- 3. Song 节点 (开口向内的对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]
    
    # 配置说明：
    # 为了让开口面向直线 (x=0)：
    # 左侧弧的圆心要在右侧，x 坐标为 (horizontal_gap + radius * cos(half_span))
    # 右侧弧的圆心要在左侧，x 坐标为 -(horizontal_gap + radius * cos(half_span))
    
    # 计算圆心位置，使得弧的端点正好落在 horizontal_gap 线上
    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {"c_x": 0, "dir": 1, "name": "Right"}, # 右侧弧，圆心在左，向右弯
        {"c_x": 0, "dir": -1, "name": "Left"}   # 左侧弧，圆心在右，向左弯
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)
        
        for row_idx, node in enumerate(col_items):
            # 角度分布
            angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span if num_in_col > 1 else 0
            
            # 计算 X: 圆心 + 方向 * (半径 * cos(角度))
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))
            
    return coords


In [7]:
# 中心直线节点间隔非线性 112度弧线
def generate_embracing_layout_with_expansion(nodes, stretch_factor=0.85):
    """
    生成开口面向直线的弧形布局
    优化：中间直线节点采用非线性分布，撑开中心间距
    stretch_factor: 拉伸系数控制】
    # 使用小于 1 的指数（如 0.5 是开根号）。
    # 指数越小，中心第一个节点与第二个节点的距离就越大，两端越拥挤。
    # 推荐值：0.5 - 0.7
    """
    coords = {}

    # --- 1. 几何参数设定 ---
    arc_radius = 800  # 半径
    arc_span = math.pi / 1.6  # 弧度张角 (约112度)
    # 弧开口端点距离中心直线的水平间距
    horizontal_gap = 300

    # 计算弧线端点的 Y 轴跨度 (用于对齐 Word)
    # y = r * sin(theta)
    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height

    # --- 2. Word 节点 (修正：中心间隔大，两端间隔小) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'],
                        key=lambda x: x['degree'],
                        reverse=True)
    num_words = len(word_nodes)

    # 长度为弧垂直跨度的 90%
    target_word_height = arc_total_height * 0.9
    half_height = target_word_height / 2

    # 【拉伸系数控制】
    # 使用小于 1 的指数（如 0.5 是开根号）。
    # 指数越小，中心第一个节点与第二个节点的距离就越大，两端越拥挤。
    # 推荐值：0.5 - 0.7
    # stretch_factor = 0.9

    max_rank = num_words // 2

    for i, node in enumerate(word_nodes):
        if i == 0:
            coords[node['id']] = (0, 0)
            continue

        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1

        # 核心逻辑：(rank / max_rank) 的 p 次方
        # 当 p < 1 时，函数图像在原点附近坡度极陡，能有效撑开中心节点
        normalized_y = (rank / max_rank)**stretch_factor * half_height

        coords[node['id']] = (0, round(direction * normalized_y, 2))

    # --- 3. Song 节点 (开口向内的对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]

    # 配置说明：
    # 为了让开口面向直线 (x=0)：
    # 左侧弧的圆心要在右侧，x 坐标为 (horizontal_gap + radius * cos(half_span))
    # 右侧弧的圆心要在左侧，x 坐标为 -(horizontal_gap + radius * cos(half_span))

    # 计算圆心位置，使得弧的端点正好落在 horizontal_gap 线上
    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {
            "c_x": 0,
            "dir": 1,
            "name": "Right"
        },  # 右侧弧，圆心在左，向右弯
        {
            "c_x": 0,
            "dir": -1,
            "name": "Left"
        }  # 左侧弧，圆心在右，向左弯
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)

        for row_idx, node in enumerate(col_items):
            # 角度分布
            angle = (arc_span / 2
                     ) - (row_idx /
                          (num_in_col - 1)) * arc_span if num_in_col > 1 else 0

            # 计算 X: 圆心 + 方向 * (半径 * cos(角度))
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))

    return coords

In [140]:
def generate_embracing_layout_with_expansion(nodes, stretch_factor=None):
    """
    生成开口面向直线的弧形布局
    优化：Word 节点按 degree 自上向下排列
    视觉：顶部（Degree大）间隔疏，向下（Degree小）间隔越密
    """
    max_degree = max(nodes, key=lambda x: x['degree'])['degree']
    if stretch_factor is None:
    # 定义阈值和对应的值（按降序排列）
        thresholds = [
            (80, 0.76),
            (70, 0.79),
            (60, 0.82),
            (50, 0.85)
        ]
        # 找到第一个满足条件的 factor，否则返回默认值 0.85
        stretch_factor = next((val for thresh, val in thresholds if max_degree > thresh), 0.85)

    coords = {}

    # --- 1. 几何参数设定 ---
    arc_radius = 800
    arc_span = math.pi / 1.6
    horizontal_gap = 300

    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height

    # --- 2. Word 节点 (顶部疏，底部密) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'],
                        key=lambda x: x['degree'],
                        reverse=True)
    num_words = len(word_nodes)

    target_word_height = arc_total_height * 0.9
    
    # stretch_factor 说明：
    # 指数越小（如 0.4-0.6），顶部节点被推开的幅度越大，底部压缩越厉害。
    # 指数 = 1.0 时，为等间距分布。

    for i, node in enumerate(word_nodes):
        # 归一化进度 t: 从 0 (顶部) 到 1 (底部)
        t = i / (num_words - 1) if num_words > 1 else 0
        
        # 核心逻辑：利用幂函数特性映射 Y 坐标
        # 我们希望在 t 较小时 y 变化快，t 较大时 y 变化慢
        # 公式：y = (t^p) * 总高度 - half_height
        # 这样当 t=0 时 y=-half_height; 当 t=1 时 y=half_height
        y_val = (t ** stretch_factor) * target_word_height - (target_word_height / 2)

        coords[node['id']] = (0, round(y_val, 2))

    # --- 3. Song 节点 (保持不变) ---
    song_nodes = [n for n in nodes if n['type'] != 'word']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]

    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {"c_x": 0, "dir": 1},  # 右侧弧
        {"c_x": 0, "dir": -1}  # 左侧弧
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)

        for row_idx, node in enumerate(col_items):
            angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span if num_in_col > 1 else 0
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))

    return coords

## 数据处理

In [141]:
def get_subset_data(df_raw, pos, words_num, is_starts_with, is_ost=False):
    # 1. 获取符合特定词性的高频词集合
    # 假设 word_count_by_pos 返回的是一个包含 'word' 列的 DataFrame
    words_set = word_count_by_pos(df_raw,
                                  pos=pos,
                                  words_num=words_num,
                                  is_starts_with=is_starts_with)
    target_words = set(words_set['word'])  # 转为 set 匹配速度更快

    # 2. 统一词性过滤逻辑
    if is_starts_with:
        mask = df_raw['pos'].str.startswith(pos, na=False)
    else:
        mask = df_raw['pos'] == pos

    # 3. 筛选、清洗并保留必要的列
    # 链式操作：过滤词性 -> 过滤高频词 -> 执行自定义清洗
    df_subset = df_raw[mask].copy()
    df_subset = df_subset[df_subset['word'].isin(target_words)]

    # 4. 统一词性标签（既然是 Subset，统一设为传入的 pos）
    df_subset['pos'] = pos
    # 统一词的id
    df_subset = id_process(df_subset, is_ost=is_ost)

    # 5. 生成年份排序映射（album_order）
    # 使用 factorize 可以直接一步生成按顺序排列的编码
    # 如果必须按年份数值排序，则先排序再 factorize
    unique_years = sorted(df_subset['song_year'].unique())
    year_to_order = {year: i for i, year in enumerate(unique_years)}
    df_subset['album_order'] = df_subset['song_year'].map(year_to_order)

    # 排序
    df_subset = df_subset.sort_values(by=['album_order'])
    return df_subset.reset_index(drop=True)

## json数据输出

In [142]:
def get_word_subset_graph(G_words_subset, df_subset, is_ost=False, stretch_factor=None):
    word_graph_dict = defaultdict(list)
    words_degree = G_words_subset.degree
    nodes_subset = []
    for n in G_words_subset.nodes():
        n_dict = {
            'id': n,
            'degree': words_degree[n],
            'type': 'word' if 'word' in n else 'song',
        }
        nodes_subset.append(n_dict)
    # pos = generate_embracing_layout(nodes_subset)
    pos =  generate_embracing_layout_with_expansion(nodes_subset, stretch_factor)
    for node in G_words_subset.nodes:
        nodes_dict = defaultdict(str)
        nodes_dict['id'] = node
        nodes_dict['size'] = words_degree[node]
        nodes_dict['x'] = pos[node][0]
        nodes_dict['y'] = pos[node][1]
        if 'word' in node:
            nodes_dict['label'] = df_subset[df_subset['word_id'] ==
                                            node]['word'].values[0]
            nodes_dict['node_type'] = 'word'
            nodes_dict['album_id'] = ""
            nodes_dict['album'] = ""
            nodes_dict['data'] = {'cluster': '词'}
        elif 'song' in node:
            nodes_dict['label'] = df_subset[df_subset['song_id_unique'] ==
                                            node]['song_name_unique'].values[0]
            nodes_dict['node_type'] = 'song'
            nodes_dict['album_id'] = df_subset[
                df_subset['song_id_unique'] ==
                node]['album_id_unique'].values[0]
            nodes_dict['album'] = df_subset[df_subset['song_id_unique'] ==
                                            node]['album_fixed'].values[0]
            nodes_dict['album_order'] = int(df_subset[
                df_subset['song_id_unique'] == node]['album_order'].values[0])
            nodes_dict['tv_name'] = ""
            if is_ost == True:
                nodes_dict['tv_name'] = df_subset[
                    df_subset['song_id_unique'] ==
                    node]['tv_name'].values[0]
            nodes_dict['data'] = {
                'cluster':
                df_subset[df_subset['song_id_unique'] == node]
                ['album_fixed'].values[0]
            }
        word_graph_dict['nodes'].append(nodes_dict)
    for edge in G_words_subset.edges:
        edges_dict = defaultdict(str)
        edges_dict['source'] = edge[0]
        edges_dict['target'] = edge[1]
        for i in edge:
            if 'song' in i:
                edges_dict['album'] = df_subset[df_subset['song_id_unique'] ==
                                                i]['album_fixed'].values[0]
                edges_dict['album_order'] = int(df_subset[
                    df_subset['song_id_unique'] == i]['album_order'].values[0])
            else:
                edges_dict['album'] = ""
                edges_dict['album_order'] = ""

        word_graph_dict['edges'].append(edges_dict)
    return word_graph_dict

# 测试

In [39]:
pos_dict = {'n': '名词', 'a': '形容词', 'v': '动词'}

In [42]:
# file_path_prefix = "data/jaychou/"
# file_path_prefix = "data/mayday/"
file_path_prefix = "data/liuyuning/"

In [43]:
df_words_raw = pd.read_csv(file_path_prefix + "cleared_words_data.csv")
df_words_raw

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,tv_name,song_name_unique,publish_date,is_ost
0,629395245,像,v,5,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,风过留痕 影视原声带,81722850,003Dq7Cs1NdHv8,265,1770084000,风过留痕,荣光,2026-02-03,1
1,629395245,光,n,4,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,风过留痕 影视原声带,81722850,003Dq7Cs1NdHv8,265,1770084000,风过留痕,荣光,2026-02-03,1
2,629395245,音乐,n,3,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,风过留痕 影视原声带,81722850,003Dq7Cs1NdHv8,265,1770084000,风过留痕,荣光,2026-02-03,1
3,629395245,抬头,v,3,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,风过留痕 影视原声带,81722850,003Dq7Cs1NdHv8,265,1770084000,风过留痕,荣光,2026-02-03,1
4,629395245,望,v,3,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,风过留痕 影视原声带,81722850,003Dq7Cs1NdHv8,265,1770084000,风过留痕,荣光,2026-02-03,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11582,581305639,故人,n,1,003w2zq32ZVlSo,烽月 (伴奏),电视剧《折腰》情感主题曲/片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,折腰 影视原声专辑,67523527,001NFTqM3zNUv2,265,1748512800,折腰,烽月 (伴奏),2025-05-29,1
11583,581305639,隐忍,v,1,003w2zq32ZVlSo,烽月 (伴奏),电视剧《折腰》情感主题曲/片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,折腰 影视原声专辑,67523527,001NFTqM3zNUv2,265,1748512800,折腰,烽月 (伴奏),2025-05-29,1
11584,581305639,痴念,v,1,003w2zq32ZVlSo,烽月 (伴奏),电视剧《折腰》情感主题曲/片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,折腰 影视原声专辑,67523527,001NFTqM3zNUv2,265,1748512800,折腰,烽月 (伴奏),2025-05-29,1
11585,581305639,止,v,1,003w2zq32ZVlSo,烽月 (伴奏),电视剧《折腰》情感主题曲/片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,折腰 影视原声专辑,67523527,001NFTqM3zNUv2,265,1748512800,折腰,烽月 (伴奏),2025-05-29,1


In [44]:
df_words = id_process(df_words_raw)

In [45]:
pos_type = "n"
df_subset = get_subset_data(df_words_raw, pos_type, words_num=50, is_starts_with=True)
df_subset

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,...,song_name_unique,publish_date,is_ost,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,word_id,album_order
0,217122344,人,n,10,0024c8zt4A4TQG,有多少爱可以重来,《江湖儿女》电影推广曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,有多少爱可以重来,2018-09-12,1,song217122344,2018,有多少爱可以重来,album4603869,有多少爱可以重来,word23,0
1,215162888,人,n,3,002OZ9wm0EhjEO,让酒,《沙海》电视剧插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,让酒,2018-07-30,1,song215162888,2018,让酒,album4254495,让酒,word23,0
2,215162888,人间,n,2,002OZ9wm0EhjEO,让酒,《沙海》电视剧插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,让酒,2018-07-30,1,song215162888,2018,让酒,album4254495,让酒,word92,0
3,215162888,路,n,2,002OZ9wm0EhjEO,让酒,《沙海》电视剧插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,让酒,2018-07-30,1,song215162888,2018,让酒,album4254495,让酒,word494,0
4,215162888,吉他,n,1,002OZ9wm0EhjEO,让酒,《沙海》电视剧插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,让酒,2018-07-30,1,song215162888,2018,让酒,album4254495,让酒,word112,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1339,629395245,命运,n,2,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,荣光,2026-02-03,1,song629395245,2026,风过留痕 影视原声带,album81722850,风过留痕 影视原声带,word34,8
1340,629395245,模样,n,2,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,荣光,2026-02-03,1,song629395245,2026,风过留痕 影视原声带,album81722850,风过留痕 影视原声带,word29,8
1341,629395245,人,n,2,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,荣光,2026-02-03,1,song629395245,2026,风过留痕 影视原声带,album81722850,风过留痕 影视原声带,word23,8
1342,629395245,音乐,n,3,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,荣光,2026-02-03,1,song629395245,2026,风过留痕 影视原声带,album81722850,风过留痕 影视原声带,word2,8


In [46]:
G_words = nx.from_pandas_edgelist(df_words, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
G_words_subset = nx.from_pandas_edgelist(df_subset, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
G_songs = nx.from_pandas_edgelist(df_words, 'song_id_unique', 'album_id_unique', edge_attr=True, create_using=nx.Graph())
G_words.number_of_nodes(), G_words_subset.number_of_nodes(), G_songs.number_of_nodes()

(4971, 194, 283)

In [47]:
# 添加数据信息
album_type = '录音室&精选辑' if file_path_prefix == "data/mayday/" else '录音室专辑'
album_type = '影视剧OST歌曲' if file_path_prefix == "data/liuyuning/" else album_type
singer = df_subset['artist_name'].values[0]
data_info = {
    'singer': singer,
    'title': f'{album_type}',
    'pos': pos_dict[pos_type],
    'all_songs_num': df_words_raw['song_id'].nunique(),
    'pos_songs_num': df_subset['song_id'].nunique(),
}
data_info

{'singer': '刘宇宁',
 'title': '影视剧OST歌曲',
 'pos': '名词',
 'all_songs_num': 145,
 'pos_songs_num': 144}

In [48]:
word_graph_dict = get_word_subset_graph(G_words_subset, df_subset)
word_graph_dict['data_info'] = data_info
with open(file_path_prefix+f'{pos_type}_word_graph_data.json', 'w', encoding='utf-8') as f:
    json.dump(word_graph_dict, f, ensure_ascii=False, indent=4)

# main

## 词-曲关系网

In [181]:
def calculate_words_songs_bip_graph(path_prefix, pos_type="n", words_num=50, is_starts_with=True, is_ost=False, stretch_factor=None):
    df_words_raw = pd.read_csv(path_prefix + "cleared_words_data.csv")
    df_subset = get_subset_data(df_words_raw, pos=pos_type, words_num=words_num, is_starts_with=is_starts_with, is_ost=is_ost)
    G_words_subset = nx.from_pandas_edgelist(df_subset, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
    # 添加数据信息
    pos_dict = {'n': '名词', 'a': '形容词', 'v': '动词'}
    album_type = '录音室&精选辑' if path_prefix == "data/mayday/" else '录音室专辑'
    album_type = '影视剧OST歌曲' if path_prefix == "data/liuyuning/" else album_type
    singer = df_subset['artist_name'].values[0]
    data_info = {
        'singer': singer,
        'title': f'{album_type}',
        'pos': pos_dict[pos_type],
        'all_num': df_words_raw['song_id'].nunique(),
        'has_pos_num': df_subset['song_id_unique'].nunique(),
        'word_num': words_num,
        'num_unit': '首',
        'nodes_type': f'{pos_dict[pos_type]}-歌曲',
    }
    word_graph_dict = get_word_subset_graph(G_words_subset, df_subset, stretch_factor)
    word_graph_dict['data_info'] = data_info
    with open(path_prefix+f'{pos_type}_word_song_graph_data.json', 'w', encoding='utf-8') as f:
        json.dump(word_graph_dict, f, ensure_ascii=False, indent=4)

In [182]:
file_path_prefix = "data/jaychou/"
# file_path_prefix = "data/mayday/"
# file_path_prefix = "data/liuyuning/"

In [183]:
for pos in ['n', 'a', 'v']:
    calculate_words_songs_bip_graph(file_path_prefix, pos_type=pos, words_num=50, is_starts_with=True, is_ost=False, stretch_factor=0.8)

## 词-专辑关系网

In [178]:
def get_word_album_subset_graph(G_words_subset,
                                df_subset,
                                stretch_factor=None):
    word_graph_dict = defaultdict(list)
    words_degree = G_words_subset.degree
    nodes_subset = []
    for n in G_words_subset.nodes():
        n_dict = {
            'id': n,
            'degree': words_degree[n],
            'type': 'word' if 'word' in n else 'album',
        }
        nodes_subset.append(n_dict)
    word_song_counts = df_subset.groupby('word_id').agg({
        'song_id': 'count',  # 计算每个词对应的不同song_id数量
    }).reset_index().rename(columns={
        'song_id': 'song_id_count',
    })
    # pos =  generate_embracing_layout_with_expansion(nodes_subset, stretch_factor)
    pos = nx.spring_layout(G_words_subset, seed=42)  # 使用 spring_layout 生成节点位置
    for node in G_words_subset.nodes:
        nodes_dict = defaultdict(str)
        nodes_dict['id'] = node
        nodes_dict['size'] = words_degree[node]
        nodes_dict['x'] = pos[node][0]
        nodes_dict['y'] = pos[node][1]
        if 'word' in node:
            nodes_dict['label'] = df_subset[df_subset['word_id'] ==
                                            node]['word'].values[0]
            nodes_dict['node_type'] = 'word'
            nodes_dict['album_id'] = ""
            nodes_dict['album'] = ""
            nodes_dict['data'] = {'cluster': '词'}
            nodes_dict['song_id_count'] = int(word_song_counts[word_song_counts['word_id'] == node]['song_id_count'].values[0])
        elif 'album' in node:
            nodes_dict['size'] = words_degree[node]
            nodes_dict['label'] = df_subset[df_subset['album_id_unique'] ==
                                            node]['album_fixed'].values[0]
            nodes_dict['node_type'] = 'album'
            nodes_dict['album_id'] = df_subset[
                df_subset['album_id_unique'] ==
                node]['album_id_unique'].values[0]
            nodes_dict['album'] = df_subset[df_subset['album_id_unique'] ==
                                            node]['album_fixed'].values[0]
            nodes_dict['album_order'] = int(df_subset[
                df_subset['album_id_unique'] == node]['album_order'].values[0])
            # nodes_dict['tv_name'] = ""
            # if is_ost == True:
            #     nodes_dict['tv_name'] = df_subset[
            #         df_subset['album_id_unique'] ==
            #         node]['tv_name'].values[0]
            nodes_dict['data'] = {
                'cluster':
                df_subset[df_subset['album_id_unique'] == node]
                ['album_fixed'].values[0]
            }
        word_graph_dict['nodes'].append(nodes_dict)
    for edge in G_words_subset.edges:
        edges_dict = defaultdict(str)
        edges_dict['source'] = edge[0]
        edges_dict['target'] = edge[1]
        edges_dict['weight'] = G_words_subset[edge[0]][edge[1]]['weight']
        for i in edge:
            if 'album' in i:
                edges_dict['album'] = df_subset[df_subset['album_id_unique'] ==
                                                i]['album_fixed'].values[0]
                edges_dict['album_order'] = int(
                    df_subset[df_subset['album_id_unique'] ==
                              i]['album_order'].values[0])
            # else:
            #     edges_dict['album'] = ""
            #     edges_dict['album_order'] = ""

        word_graph_dict['edges'].append(edges_dict)
    return word_graph_dict

In [184]:
def calculate_words_albums_bip_graph(path_prefix, pos_type="n", words_num=50, is_starts_with=True, is_ost=False, stretch_factor=None):
    df_words_raw = pd.read_csv(path_prefix + "cleared_words_data.csv")
    df_subset = get_subset_data(df_words_raw, pos=pos_type, words_num=words_num, is_starts_with=is_starts_with, is_ost=is_ost)
    # 1. 使用 groupby 统计每对节点出现的次数，并重命名为 'weight'
    df_weighted = df_subset.groupby(['word_id', 'album_id_unique']).size().reset_index(name='weight')

    # 2. 正常创建图，此时 edge_attr 会自动读取 'weight' 列
    G_words_album_subset = nx.from_pandas_edgelist(
        df_weighted, 
        'word_id', 
        'album_id_unique', 
        edge_attr='weight', 
        create_using=nx.Graph()
    )
    # 添加数据信息
    pos_dict = {'n': '名词', 'a': '形容词', 'v': '动词'}
    album_type = '录音室&精选辑' if path_prefix == "data/mayday/" else '录音室专辑'
    album_type = '影视剧OST歌曲' if path_prefix == "data/liuyuning/" else album_type
    singer = df_subset['artist_name'].values[0]
    data_info = {
        'singer': singer,
        'title': f'{album_type}',
        'pos': pos_dict[pos_type],
        'word_num': words_num,
        'all_num': df_words_raw['album_id'].nunique(),
        'has_pos_num': df_subset['album_id_unique'].nunique(),
        'num_unit': '专辑',
        'nodes_type': f'{pos_dict[pos_type]}-专辑',
    }
    word_graph_dict = get_word_album_subset_graph(G_words_album_subset, df_subset, stretch_factor=0.8)
    word_graph_dict['data_info'] = data_info
    with open(path_prefix+f'{pos_type}_word_album_graph_data.json', 'w', encoding='utf-8') as f:
        json.dump(word_graph_dict, f, ensure_ascii=False, indent=4)

In [185]:
for pos in ['n', 'a', 'v']:
    calculate_words_albums_bip_graph(file_path_prefix, pos_type=pos, words_num=15, is_starts_with=True, is_ost=False, stretch_factor=None)

In [116]:
df_subset_1 = calculate_words_albums_bip_graph(file_path_prefix, pos_type='n', words_num=15, is_starts_with=True, is_ost=False, stretch_factor=None)
df_subset_1

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,...,song_name_unique,album_id,publish_date,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,word_id,album_order
0,97756,梦,n,2,0033ZO6x09eHB0,伊斯坦堡,NaN,周杰伦,4558,0025NhlN2yWrP4,...,伊斯坦堡,8218,2000-11-07,song97756,2000,Jay,album8218,Jay,word139,0
1,97753,脸,n,2,002pImGM1R2Afa,娘子,NaN,周杰伦,4558,0025NhlN2yWrP4,...,娘子,8218,2000-11-07,song97753,2000,Jay,album8218,Jay,word878,0
2,97753,家,n,2,002pImGM1R2Afa,娘子,NaN,周杰伦,4558,0025NhlN2yWrP4,...,娘子,8218,2000-11-07,song97753,2000,Jay,album8218,Jay,word134,0
3,97758,爱情,n,6,002l8JN71d2Dxy,龙卷风,NaN,周杰伦,4558,0025NhlN2yWrP4,...,龙卷风,8218,2000-11-07,song97758,2000,Jay,album8218,Jay,word646,0
4,97753,心,n,4,002pImGM1R2Afa,娘子,NaN,周杰伦,4558,0025NhlN2yWrP4,...,娘子,8218,2000-11-07,song97753,2000,Jay,album8218,Jay,word225,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442,361947422,风,n,2,001RtHic1PivYc,错过的烟火,NaN,周杰伦,4558,0025NhlN2yWrP4,...,错过的烟火,28791467,2022-07-14,song361947422,2022,最伟大的作品,album28791467,最伟大的作品,word44,14
443,361947427,世界,n,3,003QrnHn2kR4FD,倒影,NaN,周杰伦,4558,0025NhlN2yWrP4,...,倒影,28791467,2022-07-14,song361947427,2022,最伟大的作品,album28791467,最伟大的作品,word167,14
444,361947427,眼泪,n,3,003QrnHn2kR4FD,倒影,NaN,周杰伦,4558,0025NhlN2yWrP4,...,倒影,28791467,2022-07-14,song361947427,2022,最伟大的作品,album28791467,最伟大的作品,word884,14
445,361947427,天空,n,3,003QrnHn2kR4FD,倒影,NaN,周杰伦,4558,0025NhlN2yWrP4,...,倒影,28791467,2022-07-14,song361947427,2022,最伟大的作品,album28791467,最伟大的作品,word59,14


In [136]:
# G_words_subset = nx.from_pandas_edgelist(df_subset, 'word_id', 'album_id_unique', edge_attr=True, create_using=nx.Graph())
# 1. 使用 groupby 统计每对节点出现的次数，并重命名为 'weight'
df_weighted = df_subset_1.groupby(['word_id', 'album_id_unique']).size().reset_index(name='weight')

# 2. 正常创建图，此时 edge_attr 会自动读取 'weight' 列
G_words_album_subset = nx.from_pandas_edgelist(
    df_weighted, 
    'word_id', 
    'album_id_unique', 
    edge_attr='weight', 
    create_using=nx.Graph()
)

In [144]:
get_word_album_subset_graph(G_words_album_subset, df_subset_1, stretch_factor=0.8)

defaultdict(list,
            {'nodes': [defaultdict(str,
                          {'id': 'word134',
                           'size': 14,
                           'x': 0,
                           'y': -346.23,
                           'label': '家',
                           'node_type': 'word',
                           'album_id': '',
                           'album': '',
                           'data': {'cluster': '词'}}),
              defaultdict(str,
                          {'id': 'album13004',
                           'size': 14,
                           'x': 444.46,
                           'y': 665.18,
                           'label': '依然范特西',
                           'node_type': 'album',
                           'album_id': 'album13004',
                           'album': '依然范特西',
                           'album_order': 6,
                           'data': {'cluster': '依然范特西'}}),
              defaultdict(str,
                          {'id':

In [121]:
# 按word分组计算每个词对应的song_id数量和去重song_id_unique数量
word_song_counts = df_subset_1.groupby('word').agg({
    'song_id': 'count',  # 计算每个词对应的不同song_id数量
    'song_id_unique': 'nunique',  # 计算每个词对应的不同song_id_unique数量
    'album_id_unique': 'nunique',  # 计算每个词对应的不同album_id_unique数量
    'word_id': 'nunique'
}).reset_index().rename(columns={
    'song_id': 'song_id_count',
    'song_id_unique': 'song_id_unique_count',
    'album_id_unique': 'album_id_unique_count',
    'word_id': 'word_id_count'
})
word_song_counts

,word,song_id_count,song_id_unique_count,album_id_unique_count,word_id_count
0,世界,33,33,12,1
1,人,64,64,15,1
2,天空,21,21,10,1
3,家,29,29,14,1
4,心,33,33,12,1
5,手,37,37,14,1
6,故事,25,25,13,1
7,时间,26,26,12,1
8,梦,24,24,11,1
9,爱情,20,20,13,1


In [120]:
album_word_counts = df_subset_1.groupby('album_fixed').agg({
    'word': 'count',  # 计算每个专辑对应的不同词数量
    'word_id': 'nunique',  # 计算每个专辑对应的不同词_id数量
    'song_id_unique': 'nunique'  # 计算每个专辑对应的不同song_id_unique数量
}).reset_index().rename(columns={
    'word': 'word_count',
    'word_id': 'word_id_unique_count',
    'song_id_unique': 'song_id_unique_count'
})
album_word_counts

,album_fixed,word_count,word_id_unique_count,song_id_unique_count
0,Jay,30,15,10
1,七里香,21,11,9
2,依然范特西,30,14,9
3,八度空间,32,14,10
4,十一月的萧邦,35,13,12
5,十二新作,32,14,10
6,叶惠美,31,11,11
7,周杰伦的床边故事,34,13,10
8,哎呦，不错哦,35,12,12
9,惊叹号,33,13,11


In [110]:
# 按word,album_fixed分组计算每个词对应的去重song_id_unique数量
word_album_song_counts = df_subset_1.groupby(['word', 'album_fixed']).agg({
    'song_id_unique': 'nunique'  # 计算每个词在每个专辑中对应的不同song_id_unique数量
}).reset_index().rename(columns={
    'song_id_unique': 'song_id_unique_count'
})
word_album_song_counts


,word,album_fixed,song_id_unique_count
0,世界,Jay,2
1,世界,七里香,2
2,世界,依然范特西,1
3,世界,八度空间,3
4,世界,十一月的萧邦,3
...,...,...,...
185,风,我很忙,1
186,风,最伟大的作品,1
187,风,范特西,1
188,风,跨时代,1


In [111]:
df_subset_1[(df_subset_1['album_fixed']=='最伟大的作品') & (df_subset_1['word']=='世界')]

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,...,song_name_unique,album_id,publish_date,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,word_id,album_order
417,212877900,世界,n,2,001J5QJL1pRQYB,等你下课 (with 杨瑞代),NaN,周杰伦,4558,0025NhlN2yWrP4,...,等你下课,28791467,2022-07-14,song212877900,2022,最伟大的作品,album28791467,最伟大的作品,word167,14
424,268352018,世界,n,2,001glaI72k8BQX,Mojito,NaN,周杰伦,4558,0025NhlN2yWrP4,...,Mojito,28791467,2022-07-14,song268352018,2022,最伟大的作品,album28791467,最伟大的作品,word167,14
432,361947418,世界,n,2,003w2xz20QlUZt,最伟大的作品,NaN,周杰伦,4558,0025NhlN2yWrP4,...,最伟大的作品,28791467,2022-07-14,song361947418,2022,最伟大的作品,album28791467,最伟大的作品,word167,14
437,361947426,世界,n,2,000EqWxe275Yd2,粉色海洋,NaN,周杰伦,4558,0025NhlN2yWrP4,...,粉色海洋,28791467,2022-07-14,song361947426,2022,最伟大的作品,album28791467,最伟大的作品,word167,14
440,361947422,世界,n,3,001RtHic1PivYc,错过的烟火,NaN,周杰伦,4558,0025NhlN2yWrP4,...,错过的烟火,28791467,2022-07-14,song361947422,2022,最伟大的作品,album28791467,最伟大的作品,word167,14
443,361947427,世界,n,3,003QrnHn2kR4FD,倒影,NaN,周杰伦,4558,0025NhlN2yWrP4,...,倒影,28791467,2022-07-14,song361947427,2022,最伟大的作品,album28791467,最伟大的作品,word167,14


In [113]:
df_subset_1['album_fixed'].nunique()

15

In [131]:
df_subset_1[df_subset_1['word_id'] == 'word1']

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,...,song_name_unique,album_id,publish_date,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,word_id,album_order
